# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset with the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'n/a')}")
print(f"Version: {getattr(metadata, 'version', 'n/a')}")

## 2. Data Overview
List available record sets, their fields, and columns. All entities are referenced by their `@id`.

_Note: Iterating over the record sets and fields using their `@id` as required by the Croissant schema._

In [ ]:
# List all record sets with their @id and the fields they contain
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s) in the dataset.")

for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'n/a')}")
    fields = rs.get('field', [])
    # Ensure fields is a list
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
        print(f"    - @id: {field_id}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame. Use record set and field `@id`s from the overview above.

In [ ]:
# Prepare a map of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# For illustration, we will extract the first record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Shape: {df.shape}")
    print(df.head(2))
    print()

# For further analysis, select the primary record set (first one)
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Analyzing record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, and grouping, always referencing columns by their field `@id`.

_We'll attempt numeric analysis on the primary record set, selecting the first numeric field for illustration._

In [ ]:
# Identify a numeric field in the main record set (by @id)
main_df = dataframes[main_record_set_id]

# Try to find the first numeric column
numeric_field_id = None
for col in main_df.columns:
    # Attempt to convert column to numeric; if successful, we treat as numeric
    # We'll ignore boolean/integer types first and pick float
    try:
        sample = pd.to_numeric(main_df[col], errors='coerce')
        if sample.notna().sum() > 0 and sample.dtype in [float, 'float64', 'float32', int, 'int64', 'int32']:
            numeric_field_id = col
            break
    except Exception as e:
        continue

if numeric_field_id is None:
    raise ValueError("Could not automatically identify a numeric field; please update numeric_field_id.")
else:
    print(f"Using numeric field (column @id): '{numeric_field_id}' for EDA.")
    # Convert column to numeric type
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

# FILTERING: Keep records above a threshold
threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notna().sum() > 0 else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head(2))

# NORMALIZATION
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' (first few rows):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(2))

# GROUPING: Try grouping by a string or categorical column if available
group_field_id = None
for col in main_df.columns:
    if main_df[col].dtype == object and col != numeric_field_id:
        group_field_id = col
        break

if group_field_id:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped means of '{numeric_field_id}' by '{group_field_id}':")
    print(grouped.head())
else:
    print('No suitable categorical column found for grouping.')

## 5. Visualization
Visualize distributions and relationships for selected fields using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.show()

# If grouped, show a barplot
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,4))
    order = filtered_df[group_field_id].value_counts().index[:10]
    sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, order=order)
    plt.title(f"Mean of '{numeric_field_id}' by '{group_field_id}' (top 10)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded FAIR^2 dataset metadata and explored available record sets and fields using their `@id`.
- Loaded records into DataFrames.
- Performed data filtering and normalization on numeric fields.
- Grouped values by categorical fields and visualized resulting distributions.

**This workflow can be extended for deeper analysis or additional datasets following the Croissant schema.**